# RL for LLM Training: LoRA GRPO/GSPO Walkthrough

This notebook is the interactive path through the reinforcement-learning-for-LLM-training homework. The Python scripts in this directory remain the source of truth for reward tests, sample data, metadata, and the optional Unsloth runner. Keep notebook outputs out of version control and record final results in external artifacts.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Locate the Companion Code and Record the Environment

Run this cell before any shell cells. It sets the working directory to `chapter_reinforcement_learning_llm_training` when the notebook is opened from the repository root.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


chapter_dir = find_chapter_dir("rl_lora_unsloth_reasoning.py", "chapter_reinforcement_learning_llm_training")
os.chdir(chapter_dir)


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


print("working directory:", Path.cwd())
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
for package in ["torch", "transformers", "datasets", "trl", "unsloth"]:
    print(f"{package}: {package_version(package)}")

## 2. Smoke-Test the Scaffold

These checks do not load a model. They catch syntax errors, missing optional dependencies, and reward-function regressions before a GPU run starts.

In [ ]:
%%bash
python3 -B -c 'import ast,pathlib,sys; [ast.parse(pathlib.Path(path).read_text(encoding="utf-8"), filename=path) for path in sys.argv[1:]]' rl_lora_unsloth_reasoning.py reward_functions.py unsloth_grpo_gspo_reasoning_experiment.py
python3 rl_lora_unsloth_reasoning.py --check-deps --allow-missing-deps
python3 rl_lora_unsloth_reasoning.py --unit-test-rewards
python3 unsloth_grpo_gspo_reasoning_experiment.py --check-deps --allow-missing-deps

## 3. Write Sample Data and a Metadata Template

The sample records demonstrate the expected JSONL shape. Replace or extend them before making any model-quality claim.

In [ ]:
%%bash
python3 rl_lora_unsloth_reasoning.py --write-sample-data data
python3 rl_lora_unsloth_reasoning.py --write-metadata-template data/run_metadata_template.json
python3 rl_lora_unsloth_reasoning.py --score-jsonl data/validation.jsonl

## 4. Inspect the Prompt Split

Keep validation prompts held out. Do not edit the reward after looking at validation failures unless you restart with a fresh held-out split.

In [ ]:
train_path = Path("data/train.jsonl")
validation_path = Path("data/validation.jsonl")

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_records = read_jsonl(train_path)
validation_records = read_jsonl(validation_path)
print(f"training records: {len(train_records)}")
print(f"validation records: {len(validation_records)}")
validation_records[:2]

## 5. Check the Training Note

Use versioned Unsloth GRPO or GSPO setup cells for package installation and model support. Installation commands change over time, so this repository keeps the stable checks and reward code separate from the hosted training environment.

In [ ]:
%%bash
python3 rl_lora_unsloth_reasoning.py --training-note
python3 rl_lora_unsloth_reasoning.py --training-note --use-gspo

## 6. Optional Repository GRPO Run

Run this only inside a compatible GPU environment with Unsloth and TRL installed. The runner measures a prompt-only baseline, trains a short LoRA GRPO adapter, evaluates the same held-out prompts, and writes JSON artifacts under `runs/`.

In [ ]:
RUN_GRPO_EXPERIMENT = False

if RUN_GRPO_EXPERIMENT:
    subprocess.run(
        [
            sys.executable,
            "unsloth_grpo_gspo_reasoning_experiment.py",
            "--run-experiment",
            "--algorithm",
            "grpo",
            "--artifact-dir",
            "runs/unsloth_grpo_reasoning",
            "--model-name",
            "Qwen/Qwen2.5-0.5B-Instruct",
            "--max-steps",
            "5",
            "--num-generations",
            "4",
        ],
        check=True,
    )
else:
    print("Skipped. Set RUN_GRPO_EXPERIMENT = True inside a compatible GPU runtime.")

## 7. Optional One-Factor GSPO Ablation

This cell changes only the algorithm from GRPO to GSPO. Keep the model, seed, generation settings, LoRA settings, and step budget matched unless the ablation question is different.

In [ ]:
RUN_GSPO_ABLATION = False

if RUN_GSPO_ABLATION:
    subprocess.run(
        [
            sys.executable,
            "unsloth_grpo_gspo_reasoning_experiment.py",
            "--run-experiment",
            "--algorithm",
            "gspo",
            "--artifact-dir",
            "runs/unsloth_gspo_reasoning",
            "--model-name",
            "Qwen/Qwen2.5-0.5B-Instruct",
            "--max-steps",
            "5",
            "--num-generations",
            "4",
        ],
        check=True,
    )
else:
    print("Skipped. Set RUN_GSPO_ABLATION = True after the baseline GRPO run is working.")

## 8. Summarize Artifacts for the Report

Point `summary_paths` at the completed run summaries. The report should compare held-out score, invalid-output rate, average generated length, runtime, throughput, and peak memory.

In [ ]:
summary_paths = [
    Path("runs/unsloth_grpo_reasoning/unsloth_grpo_gspo_summary.json"),
    Path("runs/unsloth_gspo_reasoning/unsloth_grpo_gspo_summary.json"),
]

for path in summary_paths:
    if not path.exists():
        print(f"missing: {path}")
        continue

    summary = json.loads(path.read_text(encoding="utf-8"))
    baseline = summary.get("baseline_metrics", {})
    trained = summary.get("trained_metrics", {})
    training = summary.get("training_summary", {})
    peak = summary.get("peak_memory", {}).get("training", {})

    print("\n", path)
    print("algorithm:", summary.get("algorithm"))
    print("model:", summary.get("model_name"))
    print("baseline exact:", baseline.get("exact_answer_accuracy"))
    print("trained exact:", trained.get("exact_answer_accuracy"))
    print("trained invalid rate:", trained.get("invalid_output_rate"))
    print("trained average tokens:", trained.get("average_generated_tokens"))
    print("training seconds:", training.get("training_seconds"))
    print("training peak reserved GiB:", peak.get("reserved_gib"))

## 9. Qualitative Examples

Inspect the output JSONL and choose one improvement, one failure, and one case where the reward was misleading or incomplete.

In [ ]:
outputs_path = Path("runs/unsloth_grpo_reasoning/unsloth_grpo_gspo_outputs.jsonl")

if outputs_path.exists():
    rows = read_jsonl(outputs_path)
    for row in rows[:6]:
        print("\nphase:", row.get("phase"))
        print("question:", row.get("question"))
        print("expected:", row.get("expected"))
        print("generated:", row.get("generated_text"))
        print("reward:", row.get("reward"))
else:
    print(f"No output artifact found at {outputs_path}.")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.